<a href="https://colab.research.google.com/github/ManasaRPatel/MachineLearning2_LAB/blob/main/candidate_elimination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

data = {
    'CGPA': ['High', 'High', 'Medium', 'Low', 'High', 'Medium'],
    'Attendance': ['High', 'High', 'High', 'Low', 'High', 'High'],
    'Backlogs': ['No', 'No', 'No', 'Yes', 'Yes', 'No'],
    'Income': ['Low', 'Medium', 'Low', 'High', 'Low', 'Medium'],
    'Eligible': ['Yes', 'Yes', 'Yes', 'No', 'No', 'Yes']
}

df = pd.DataFrame(data)
df.to_csv('scholarship.csv', index=False)

# Read the dataset
df = pd.read_csv('scholarship.csv')

display(df)

,CGPA,Attendance,Backlogs,Income,Eligible
0,High,High,No,Low,Yes
1,High,High,No,Medium,Yes
2,Medium,High,No,Low,Yes
3,Low,Low,Yes,High,No
4,High,High,Yes,Low,No
5,Medium,High,No,Medium,Yes


In [2]:
import pandas as pd
from itertools import product

# Load the dataset
df = pd.read_csv('scholarship.csv')

attributes = list(df.columns[:-1])
domains = [df[col].unique().tolist() for col in attributes]

n = len(attributes)
ANY = '?'
NULL = 'Ø'

def covers(h, x):
    return all(
        hv == ANY or hv == xv
        for hv, xv in zip(h, x)
    )

def more_general_or_equal(h1, h2):
    return all(
        b == NULL or a == ANY or a == b
        for a, b in zip(h1, h2)
    )

def generalize_minimally(s, x):
    result = list(s)
    for i in range(n):
        if result[i] == NULL:
            result[i] = x[i]
        elif result[i] != x[i]:
            result[i] = ANY
    return tuple(result)

def specialize_minimally(g, x, S):
    result = []

    for i in range(n):
        if g[i] == ANY:
            for value in domains[i]:
                if value != x[i]:
                    h = list(g)
                    h[i] = value
                    h = tuple(h)

                    if any(more_general_or_equal(h, s) for s in S):
                        result.append(h)

    return result

# Initialize boundaries
S = {(NULL,) * n}
G = {(ANY,) * n}

print("Initial S:", S)
print("Initial G:", G)

for _, row in df.iterrows():
    x = tuple(row[attributes])
    label = row['Eligible']

    if label == 'Yes':
        # Remove G hypotheses that do not cover the positive example
        G = {g for g in G if covers(g, x)}

        # Generalize inconsistent S hypotheses
        new_S = set()

        for s in S:
            if covers(s, x):
                new_S.add(s)
            else:
                h = generalize_minimally(s, x)
                if any(more_general_or_equal(g, h) for g in G):
                    new_S.add(h)

        S = {
            s for s in new_S
            if not any(
                s != other and more_general_or_equal(s, other)
                for other in new_S
            )
        }

    else:
        # Remove S hypotheses that cover the negative example
        S = {s for s in S if not covers(s, x)}

        # Specialize inconsistent G hypotheses
        new_G = set()

        for g in G:
            if not covers(g, x):
                new_G.add(g)
            else:
                new_G.update(specialize_minimally(g, x, S))

        G = {
            g for g in new_G
            if not any(
                g != other and more_general_or_equal(other, g)
                for other in new_G
            )
        }

    print(f"\nExample: {x}, Label: {label}")
    print("S:", S)
    print("G:", G)

print("\nFinal Specific Boundary S:")
for h in S:
    print(h)

print("\nFinal General Boundary G:")
for h in G:
    print(h)

Initial S: {('Ø', 'Ø', 'Ø', 'Ø')}
Initial G: {('?', '?', '?', '?')}

Example: ('High', 'High', 'No', 'Low'), Label: Yes
S: {('High', 'High', 'No', 'Low')}
G: {('?', '?', '?', '?')}

Example: ('High', 'High', 'No', 'Medium'), Label: Yes
S: {('High', 'High', 'No', '?')}
G: {('?', '?', '?', '?')}

Example: ('Medium', 'High', 'No', 'Low'), Label: Yes
S: {('?', 'High', 'No', '?')}
G: {('?', '?', '?', '?')}

Example: ('Low', 'Low', 'Yes', 'High'), Label: No
S: {('?', 'High', 'No', '?')}
G: {('?', 'High', '?', '?'), ('?', '?', 'No', '?')}

Example: ('High', 'High', 'Yes', 'Low'), Label: No
S: {('?', 'High', 'No', '?')}
G: {('?', '?', 'No', '?')}

Example: ('Medium', 'High', 'No', 'Medium'), Label: Yes
S: {('?', 'High', 'No', '?')}
G: {('?', '?', 'No', '?')}

Final Specific Boundary S:
('?', 'High', 'No', '?')

Final General Boundary G:
('?', '?', 'No', '?')
